In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEndpoint, ChatHuggingFace
from dotenv import load_dotenv
from typing import TypedDict, Annotated, NotRequired
from pydantic import BaseModel, Field
import json
import operator
import re

In [2]:
class strModel(BaseModel):
    feedback: str = Field(..., description="feedback from user")
    score: int = Field(..., description="score from user", ge=0, le=10)

In [ ]:

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen3-32B",
    task="text-generation",
    temperature=0.7,
)
chatModel=ChatHuggingFace(llm=llm)


structure_model = chatModel.with_structured_output(
    strModel,
    method="json_schema"
)

In [ ]:
class WorkflowState(TypedDict):
    essay: str
    language_feedback: NotRequired[str]
    cot_feedback: NotRequired[str]
    analysis_feedback: NotRequired[str]
    overall_feedback: NotRequired[str]
    individual_scores: NotRequired[Annotated[list[int], operator.add]]
    avg_score: NotRequired[float]


In [ ]:
essay="""
# The Impact of Artificial Intelligence on Software Development

Artificial Intelligence has become one of the most influential technologies in modern software development. In the past, software developers were responsible for manually writing almost every part of an application, from designing algorithms to debugging errors. Today, AI-powered tools are changing the way developers design, build, test, and maintain software. Rather than replacing software developers, artificial intelligence is increasingly becoming a powerful assistant that helps them work faster and solve more complex problems.

One of the most significant applications of AI in software development is code generation. Modern AI systems can understand natural-language instructions and generate code in programming languages such as Python, JavaScript, Java, and C++. A developer can describe a function or feature and receive a possible implementation within seconds. This can save considerable time when creating repetitive code, writing utility functions, or learning unfamiliar technologies. However, developers still need to understand the generated code because AI-generated solutions can contain logical errors, security vulnerabilities, or inefficient approaches.

Artificial intelligence is also useful for debugging. Finding the cause of an error can sometimes take longer than writing the original code. AI assistants can analyze error messages, stack traces, and source code to suggest possible causes of a problem. They can also explain complicated errors in simpler language. This is particularly useful for beginners who may not immediately understand technical error messages. Experienced developers can also benefit because AI can help them investigate large codebases more quickly.

Another important area is software testing. Testing is essential for ensuring that an application works correctly, but manually creating test cases can be time-consuming. AI systems can help developers generate unit tests, identify edge cases, and analyze areas of an application that may require additional testing. Nevertheless, human judgment remains important because developers must determine whether the generated tests actually represent the application's business requirements.

AI is also changing how developers interact with technical knowledge. Traditionally, developers searched documentation, tutorials, forums, and search engines whenever they encountered an unfamiliar problem. AI assistants can provide explanations based on a developer's specific question and context. This makes learning new frameworks and programming concepts faster. However, official documentation and reliable technical sources remain important because AI-generated information can occasionally be outdated or incorrect.

Generative AI has introduced another major area of software engineering: applications powered by Large Language Models. Developers can now build systems that understand and generate natural language. Techniques such as Retrieval-Augmented Generation allow applications to retrieve information from external documents before generating an answer. This makes it possible to develop intelligent assistants for education, customer support, research, healthcare information, and enterprise knowledge management.

AI agents are extending these capabilities further. Instead of simply answering a question, an agent can potentially decide which actions are required, use available tools, retrieve information, and maintain state across multiple steps. Frameworks for building stateful AI workflows allow developers to control how these decisions are made. This is important because production AI applications often require more than a single request to a language model.

Despite these advantages, artificial intelligence introduces important challenges. AI systems can generate incorrect information, commonly referred to as hallucination. They may also produce insecure or inefficient code. Privacy is another concern when sensitive company information is provided to external AI systems. Developers therefore need mechanisms for validation, security, monitoring, evaluation, and human oversight.

The role of software developers is consequently evolving rather than disappearing. Developers increasingly need to understand not only how to write code but also how to design systems in which traditional software components and AI models work together. Skills such as problem solving, system design, database design, security, testing, and software architecture remain extremely important because AI-generated components still have to operate inside reliable software systems.

In conclusion, artificial intelligence is transforming software development by assisting with coding, debugging, testing, learning, and the creation of intelligent applications. It can significantly increase developer productivity, but it does not eliminate the need for strong engineering knowledge. Developers who understand both traditional software engineering and modern AI technologies will be better positioned to build reliable and useful applications. The future of software development is therefore likely to involve collaboration between human developers and increasingly capable AI systems.

"""

In [ ]:
def get_language_feedback(state: WorkflowState) -> WorkflowState:
    essay = state["essay"]
    prompt = f"Provide language feedback for this essay: {essay}. Also give a score from 1 to 10."
    output = get_structured_feedback(prompt)

    return {"language_feedback": output.feedback, "individual_scores": [output.score]}

In [ ]:
def get_cot_feedback(state: WorkflowState) -> WorkflowState:
    essay = state["essay"]
    prompt = f"Provide clarity-of-thought feedback for this essay: {essay}. Also give a score from 1 to 10."
    output = get_structured_feedback(prompt)

    return {"cot_feedback": output.feedback, "individual_scores": [output.score]}

In [ ]:
def get_analysis_feedback(state: WorkflowState) -> WorkflowState:
    essay = state["essay"]
    prompt = f"Provide analysis feedback for this essay: {essay}. Also give a score from 1 to 10."
    output = get_structured_feedback(prompt)

    return {"analysis_feedback": output.feedback, "individual_scores": [output.score]}

In [ ]:
def get_overall_feedback(state: WorkflowState) -> WorkflowState:
    language_feedback = state["language_feedback"]
    cot_feedback = state["cot_feedback"]
    analysis_feedback = state["analysis_feedback"]
    individual_scores = state["individual_scores"]

    avg_score = sum(individual_scores) / len(individual_scores)

    overall_prompt = (
        "Provide overall feedback for this essay based on the following feedbacks: "
        f"{language_feedback}, {cot_feedback}, {analysis_feedback}. "
        f"The average score is {avg_score:.2f}/10."
    )
    output = chatModel.invoke(overall_prompt).content

    return {"overall_feedback": output, "avg_score": avg_score}

In [ ]:
state_graph=StateGraph(WorkflowState)


state_graph.add_node("get_language_feedback",get_language_feedback)
state_graph.add_node("get_cot_feedback",get_cot_feedback)
state_graph.add_node("get_analysis_feedback",get_analysis_feedback)
state_graph.add_node("get_overall_feedback",get_overall_feedback)


state_graph.add_edge(START, "get_language_feedback")
state_graph.add_edge(START, "get_cot_feedback")
state_graph.add_edge(START, "get_analysis_feedback")
state_graph.add_edge("get_language_feedback" , "get_overall_feedback")
state_graph.add_edge("get_cot_feedback" , "get_overall_feedback")
state_graph.add_edge("get_analysis_feedback" , "get_overall_feedback")

state_graph.add_edge("get_overall_feedback", END)


workflow=state_graph.compile()


In [ ]:
initial_state = {
    'essay': essay
}

workflow.invoke(initial_state)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)